In [2]:
import pandas as pd

df_ground_truth = pd.read_csv("../data/ground_truth.csv")
ground_truth = df_ground_truth.to_dict(orient="records")

In [3]:
ground_truth[10]

{'question': 'Will students get a Zoom link to join the live office hours?',
 'document': '489dd1c9d9'}

In [4]:
from ingest import load_faq_data, build_index

documents = load_faq_data()

documents_llm = []

for doc in documents:
    if doc["course"] == "llm-zoomcamp":
        documents_llm.append(doc)

documents = documents_llm
index = build_index(documents)

In [5]:
doc_idx = {}

for doc in documents:
    doc_idx[doc["id"]] = doc

In [6]:
q = ground_truth[10]
q

{'question': 'Will students get a Zoom link to join the live office hours?',
 'document': '489dd1c9d9'}

In [7]:
doc_idx[q['document']]

{'id': '489dd1c9d9',
 'course': 'llm-zoomcamp',
 'section': 'General Course-Related Questions',
 'question': 'What is the video/zoom link to the stream for the “Office Hours” or live/workshop sessions?',
 'answer': 'The zoom link is only published to instructors/presenters/TAs.\n\nStudents participate via YouTube Live and submit questions to Slido (link is pinned in the chat when live). The video URL should be posted in the [announcements channel on Telegram and Slack](https://t.me/dezoomcamp) before it begins. You can also watch live on the DataTalksClub [YouTube Channel](https://www.youtube.com/c/DataTalksClub).\n\nDon’t post questions in chat as they may be missed if the room is very active.'}

In [9]:
import os
from dotenv import load_dotenv
from google import genai

load_dotenv()

client = genai.Client(
api_key=os.getenv("GOOGLE_API_KEY")
)


In [11]:
from evaluation_utils import RAGWithUsage

assistant = RAGWithUsage(
    index=index,
    llm_client=client,
    course='llm-zoomcamp',
)

In [12]:
q['question']

'Will students get a Zoom link to join the live office hours?'

In [13]:
answer = assistant.rag(q['question'])

In [14]:
assistant.total_cost()

0.0003109

In [15]:
print(answer)

No, students will not get a Zoom link to join the live office hours. The Zoom link is only published to instructors, presenters, and TAs. Students participate via YouTube Live and submit questions to Slido.


In [16]:
doc_id = q["document"]
original_doc = doc_idx[doc_id]
answer_orig = original_doc["answer"]

answer_orig

'The zoom link is only published to instructors/presenters/TAs.\n\nStudents participate via YouTube Live and submit questions to Slido (link is pinned in the chat when live). The video URL should be posted in the [announcements channel on Telegram and Slack](https://t.me/dezoomcamp) before it begins. You can also watch live on the DataTalksClub [YouTube Channel](https://www.youtube.com/c/DataTalksClub).\n\nDon’t post questions in chat as they may be missed if the room is very active.'

In [17]:
rag_result = {
    "question": q['question'],
    "answer_llm": answer,
    "answer_orig": answer_orig,
    "document": doc_id,
}

rag_result

{'question': 'Will students get a Zoom link to join the live office hours?',
 'answer_llm': 'No, students will not get a Zoom link to join the live office hours. The Zoom link is only published to instructors, presenters, and TAs. Students participate via YouTube Live and submit questions to Slido.',
 'answer_orig': 'The zoom link is only published to instructors/presenters/TAs.\n\nStudents participate via YouTube Live and submit questions to Slido (link is pinned in the chat when live). The video URL should be posted in the [announcements channel on Telegram and Slack](https://t.me/dezoomcamp) before it begins. You can also watch live on the DataTalksClub [YouTube Channel](https://www.youtube.com/c/DataTalksClub).\n\nDon’t post questions in chat as they may be missed if the room is very active.',
 'document': '489dd1c9d9'}

In [18]:
def generate_rag_answer(rec):
    question = rec["question"]
    doc_id = rec["document"]
    original_doc = doc_idx[doc_id]

    answer_llm = assistant.rag(question)
    answer_orig = original_doc["answer"]

    result = {
        "question": question,
        "answer_llm": answer_llm,
        "answer_orig": answer_orig,
        "document": doc_id,
    }

    return result

In [19]:
record = generate_rag_answer(q)
record

{'question': 'Will students get a Zoom link to join the live office hours?',
 'answer_llm': 'No, students will not get a Zoom link to join the live office hours. The Zoom link is only published to instructors, presenters, and TAs. Students participate via YouTube Live.',
 'answer_orig': 'The zoom link is only published to instructors/presenters/TAs.\n\nStudents participate via YouTube Live and submit questions to Slido (link is pinned in the chat when live). The video URL should be posted in the [announcements channel on Telegram and Slack](https://t.me/dezoomcamp) before it begins. You can also watch live on the DataTalksClub [YouTube Channel](https://www.youtube.com/c/DataTalksClub).\n\nDon’t post questions in chat as they may be missed if the room is very active.',
 'document': '489dd1c9d9'}

In [20]:
assistant.total_cost()

0.0006068

In [21]:
assistant.reset_usage()

In [22]:
assistant.total_cost()

0

In [23]:
from concurrent.futures import ThreadPoolExecutor
from evaluation_utils import map_progress

In [24]:
with ThreadPoolExecutor(max_workers=6) as pool:
    results = map_progress(pool, ground_truth, generate_rag_answer)

  0%|          | 0/575 [00:00<?, ?it/s]

In [25]:
results[:10]

[{'question': "Is it possible to enroll in the course even if I've just found it?",
  'answer_llm': "Yes, you can still join the course even if you've just discovered it. However, if you wish to receive a certificate, you will need to submit your project while submissions are still being accepted.",
  'answer_orig': 'Yes, but if you want to receive a certificate, you need to submit your project while we’re still accepting submissions.',
  'document': '74eb249bbf'},
 {'question': 'What are the requirements for earning a course certificate?',
  'answer_llm': 'To earn a course certificate, you must:\n\n*   Finish the course with a "live" cohort.\n*   Finish a capstone project.\n*   Complete the required peer reviews.\n*   Project submission and peer review must happen while a live cohort is accepting them.\n\nHomework is not required to get the certificate.',
  'answer_orig': 'Yes, but if you want to receive a certificate, you need to submit your project while we’re still accepting submis

In [26]:
df_results = pd.DataFrame(results)

In [27]:
df_results.head()

,question,answer_llm,answer_orig,document
0,Is it possible to enroll in the course even if...,"Yes, you can still join the course even if you...","Yes, but if you want to receive a certificate,...",74eb249bbf
1,What are the requirements for earning a course...,"To earn a course certificate, you must:\n\n* ...","Yes, but if you want to receive a certificate,...",74eb249bbf
2,Is there a specific deadline for submitting th...,"To get a certificate, project submission and p...","Yes, but if you want to receive a certificate,...",74eb249bbf
3,"If I join the course late, can I still qualify...","Yes, you can still join the course late. Howev...","Yes, but if you want to receive a certificate,...",74eb249bbf
4,Are there any time constraints for project sub...,"Yes, there are time constraints for project su...","Yes, but if you want to receive a certificate,...",74eb249bbf


In [28]:
assistant.total_cost()

0.3301332999999998

In [30]:
df_results.to_csv("../data/rag-answers-new.csv", index=False)